# Showcase of the Multichannel Remote Controller QCoDeS driver(v0.1.0)
Copyright (c) Basel Precision Instruments AG (2026), written for the QCoDeS driver Baspi_Mcrc.py, v0.1.0

..................................................................................................................................................................................................................


This notebook shows brief examples on how to use the QCoDeS driver for the Basel Precision Instruments Multichannel Remote Controller.

### Hardware Setup

Before using the driver, ensure the physical amplifiers are set to **Remote** mode:

| Device | GAIN Switch | fcut Switch |
|--------|-------------|-------------|
| I/V Converter (SP983c) | `1E5, Remote` | `30Hz, Remote` |
| Differential Amplifier (SP1004) | `x1000, Remote` | `100Hz, Remote` |

## 1 - Imports and setting up a connection to the device

The main driver class is the `BaspiMcrc` class. It contains all the important functions to use the Multichannel Remote Controller. It allows to add and remove channels, change the gain and the cutoff frequency and ask the status of the device, <br>
read if the device is overloaded (for `I to V Converter` and `Differential Amplifier`) or compensated (only `Differential Amplifier`) close the instrument and a reconnect method, in case the connestion is lost while using it.

In [ ]:
from Baspi_Mcrc import BaspiMcrc

mcrc = BaspiMcrc(
    name='mcrc',
    host='192.168.178.50',
    coldstart='KEEP', #DEFAULT= resets the DB and sets all devices to the default settings / KEEP = keeps the last settings which were saved before shutting down the Multichannel Remote Controller
    username="admin",
    password="admin"
)

## 2 - Add and remove channels

To set and change the gain and cutoff frequency, the channels / devices need to be added first. Be aware that even if it is "added" on the web interface, the driver needs it to be added manually in the driver first, before configurating it. This minimizes any possible mix-up between actually added and used channel. For further clarifications, a dedicated name can be given to each device, which will be displayed on the web interface. The parameters can not be set, if the corresponding channel is not added.

Following are the dedicated channels for the I to V Converters and Differential Amplifiers:

| `I to V Converter` | `Differential Amplifier`  |
| :--------: | :--------: |
| IV1 | DA1 |
| IV2 | DA2 |
| IV3 | DA3 |
| IV4 | DA4 |




### 2.1 - Add

In [ ]:
mcrc.add_channel("da1", "Differential Amplifier")
mcrc.add_channel("iv1")

mcrc.add_channel("iv2")

### 2.2 - Remove

In [ ]:
mcrc.remove_channel("iv2")

## 3 - Set Parameters

After adding the channel, changing and setting parameters is possible with the set command. Reading the current settings can be done by using either the get-function or with the status function.

### I/V Converter (SP983)

| Parameter | Valid Values | Unit |
|-----------|--------------|------|
| **Gain** | `1E5`, `1E6`, `1E7`, `1E8`, `1E9` | V/A |
| **Cutoff** | `30`, `100`, `300`, `1K`, `3K`, `10K`, `30K`, `100K`, `1M` | Hz |

### Differential Amplifier (SP1004)

| Parameter | Valid Values | Unit |
|-----------|--------------|------|
| **Gain** | `1E2`, `1E3`, `1E4` | V/V |
| **Cutoff** | `100`, `300`, `1K`, `3K`, `10K`, `30K`, `100K`, `300K`, `1M` | Hz |

In [ ]:
print(f"IV1 Gain: {mcrc.iv1.gain()}")
print(f"IV1 Cutoff: {mcrc.iv1.cutoff()}")

mcrc.iv1.gain('1E9')
print(f"New gain: {mcrc.iv1.gain()}")
mcrc.iv1.cutoff('10K')
print(f"IV1 Cutoff: {mcrc.iv1.cutoff()}")

mcrc.da1.gain("1E4")
print(mcrc.da1.gain.get())
mcrc.da1.cutoff("10K")
print(mcrc.da1.cutoff.get())


In [ ]:
mcrc.iv2.gain('1E7')
mcrc.iv2.cutoff("1M")

In [ ]:
mcrc.iv1.get_status()

### 3.1 - Set parameters without adding a channel

When trying to set any parameters without adding the channel before, you will meet the error message as following:

In [ ]:
mcrc.iv2.gain.set("1E5")
mcrc.iv2.cutoff("1K")
print(mcrc.iv2.cutoff.get())

### 4 - Reconnecting

If the connection is cut unexpectedly, the `reconnect()` function allows the device to reconnect to the current running session:<br>
<br>
`attempts` allows the user to choose how many reconnection attempts should happen, before the session gets closed.<br>
`wait_between_attempts` allows the user to choose how many seconds should pass before a new try.

In [ ]:
mcrc.reconnect(attempts= 20, wait_between_attempts= 3)

# 5 - Closing the session

`close()` is used to suspend the current session. This also removes the Instrument. To reconnect to the Multichannel Remote Controller, step 1 and 2 have to be repeated!

In [ ]:
mcrc.close()